# 📄 DocIQ — Installation Guide

DocIQ is a Streamlit-based RAG (Retrieval-Augmented Generation) document intelligence platform. Upload documents, ask questions in natural language, and get answers with source citations — powered by LangChain, ChromaDB, sentence-transformers, and multiple LLM providers (Gemini, DeepSeek, Llama 3.3).

This notebook walks through a full local installation, from prerequisites to first run, plus optional cloud deployment.

**Contents**
1. Prerequisites
2. Get the project files
3. Create a virtual environment
4. Install dependencies
5. Configure API keys (`.env`)
6. Run the app
7. First-run notes (embedding model, ChromaDB)
8. Authentication (`users.json`)
9. Troubleshooting
10. Optional: Deploy to Streamlit Community Cloud


## 1. Prerequisites

Make sure you have the following installed before starting:

| Requirement | Notes |
|---|---|
| **Python 3.10+** | Check with `python --version`. 3.10–3.12 recommended. |
| **pip** | Comes bundled with Python. |
| **Git** *(optional)* | Only needed if cloning from a repository. |
| **~1 GB free disk space** | For dependencies + the embedding model cache. |
| **Internet connection** | Needed once, to download packages and the embedding model. After that, everything runs offline except LLM calls. |

**Hardware notes (CPU-only machines, 8GB RAM):**
DocIQ is built to run comfortably on a CPU-only, 8GB-RAM machine:
- Embedding model: `all-MiniLM-L6-v2` — only ~80MB, fast on CPU, no GPU required.
- ChromaDB (vector storage) runs locally, no external database needed.
- No local LLM is loaded — all chat models are called via API, so your machine only handles embeddings + retrieval, not generation.


## 2. Get the project files

If you already have the project folder, skip to Step 3.

**Option A — Clone from Git**
```bash
git clone <your-repo-url> dociq
cd dociq
```

**Option B — Manual copy**
Place the following files in a single project folder (e.g. `dociq/`):

```
dociq/
├── app.py
├── auth.py
├── cost_estimator.py
├── document_processor.py
├── langchain_pipeline.py
├── providers.py
├── rag_engine.py
├── style.py
├── vector_store.py
├── requirements.txt
├── README.md
└── .env          (you'll create this in Step 5)
```


## 3. Create a virtual environment

Isolating dependencies in a virtual environment avoids version conflicts with other Python projects.

**Windows (PowerShell / CMD)**
```bash
python -m venv venv
venv\Scripts\activate
```

**macOS / Linux**
```bash
python3 -m venv venv
source venv/bin/activate
```

You'll know it worked if your terminal prompt now shows `(venv)` at the start of the line.

> 💡 Every time you come back to work on DocIQ in a new terminal session, re-run the `activate` command above before doing anything else.


## 4. Install dependencies

With the virtual environment active, install everything from `requirements.txt`:

```bash
pip install -r requirements.txt
```

This installs:

| Package | Purpose |
|---|---|
| `streamlit` | Web UI framework |
| `python-dotenv` | Loads API keys from `.env` |
| `requests` | Gemini REST API calls |
| `openai` | OpenAI-compatible client (used for GitHub Models / DeepSeek / Llama) |
| `langchain`, `langchain-core`, `langchain-community` | Conversational retrieval chain + memory |
| `langchain-text-splitters` | Document chunking |
| `chromadb` | Local vector database |
| `sentence-transformers` | Embedding model (`all-MiniLM-L6-v2`) |
| `pypdf` | PDF text extraction |
| `python-docx` | DOCX text extraction |
| `rank-bm25` | Keyword search half of hybrid retrieval |
| `pydantic` | Data validation (used by LangChain wrappers) |

**Expect this step to take a few minutes** the first time — `chromadb` and `sentence-transformers` pull in several sub-dependencies.

> ⚠️ If `pip install` fails on Windows for a specific package (commonly `chromadb`'s native dependencies), make sure you're on a supported Python version (3.10–3.12) and that pip itself is up to date: `python -m pip install --upgrade pip`.


## 5. Configure API keys (`.env`)

DocIQ needs at least one LLM provider key to actually answer questions (retrieval/embeddings work without any key, but chat generation doesn't).

Create a file named `.env` in the project root:

```bash
# Windows
type nul > .env

# macOS / Linux
touch .env
```

Add the keys you have. You don't need all three — the app works with just one provider configured.

```ini
# .env
GEMINI_API_KEY=your_real_gemini_key
GITHUB_TOKEN=your_real_github_token
OPENROUTER_API_KEY=your_real_openrouter_key
```

**Where to get each key (free tiers available):**

| Provider | Get a key at | Notes |
|---|---|---|
| **Gemini** | [aistudio.google.com/apikey](https://aistudio.google.com/apikey) | Powers "Gemini 3.5 Flash" |
| **GitHub Models** | [github.com/settings/tokens](https://github.com/settings/tokens) | Classic token with `models: read` scope. Powers "DeepSeek R1" and "Llama 3.3 70B" — free during preview |
| **OpenRouter** | [openrouter.ai/keys](https://openrouter.ai/keys) | Powers "Claude 3.5 Sonnet", "GPT-4 Turbo", "Mistral 7B" |

> 🔒 **Never commit `.env` to version control.** It should already be listed in `.gitignore`. Treat every key in it as a secret.


## 6. Run the app

With the virtual environment active and `.env` configured:

```bash
streamlit run app.py
```

Streamlit will print a local URL, typically:

```
Local URL: http://localhost:8501
```

Open it in your browser. You should land on the sign-up / log-in screen.


## 7. First-run notes

**Embedding model download**
The very first time you run the app (or upload a document), it downloads `all-MiniLM-L6-v2` (~80MB) from Hugging Face. This needs an internet connection once; afterward it's cached locally at:

- Windows: `C:\Users\<you>\.cache\huggingface`
- macOS/Linux: `~/.cache/huggingface`

**Local persistence**
Two things are created automatically in your project folder on first use:

| Path | What it stores | Persists across restarts? |
|---|---|---|
| `./chroma_db/` | Document embeddings (ChromaDB) | ✅ Yes, locally |
| `./users.json` | Account records (PBKDF2 password hashes) | ✅ Yes, locally |

Both are plain local files — nothing is sent externally except the LLM API calls themselves.


## 8. Authentication

DocIQ uses real local authentication (`auth.py`) — not a name-only demo gate:

- Passwords are hashed with **PBKDF2-HMAC-SHA256** (200,000 iterations) plus a random per-user salt — never stored or compared in plaintext.
- Accounts are stored in `users.json` in the project root.
- Each user gets an isolated document collection, so uploads and chats don't mix between accounts.

**First-time use:** open the app, choose **Sign up**, pick a username (3–32 characters: letters, numbers, `_`, `.`, `-`) and a password (8+ characters, mixed case, at least one digit).

> ⚠️ `users.json` holds password hashes, not plaintext — but treat it as a secret file anyway. Don't commit it to a public repo.


## 9. Troubleshooting

| Symptom | Likely cause / fix |
|---|---|
| `ModuleNotFoundError` on launch | Virtual environment isn't active, or `pip install -r requirements.txt` didn't complete. Re-activate `venv` and re-run install. |
| App says "Missing Gemini API key" / similar | The corresponding key isn't set in `.env`, or `.env` isn't in the project root. |
| First upload is slow | Normal — it's downloading the embedding model. Subsequent uploads are fast. |
| "No extractable text found" on a PDF | The PDF is likely scanned/image-only. OCR isn't implemented; use a text-based PDF instead. |
| Port `8501` already in use | Another Streamlit app is running. Either close it or run `streamlit run app.py --server.port 8502`. |
| Documents/chats disappear after restart | You're likely running on an ephemeral filesystem (e.g. Streamlit Community Cloud's free tier resets `chroma_db/` and `users.json` on redeploy). For persistence there, point `PERSIST_DIR` (`vector_store.py`) and `USERS_FILE` (`auth.py`) at a mounted volume, or use a real database. |
| `pip install` fails on `chromadb` (Windows) | Upgrade pip (`python -m pip install --upgrade pip`) and confirm Python 3.10–3.12. |


## 10. Optional: Deploy to Streamlit Community Cloud (free)

1. Push the project to a GitHub repo. `.gitignore` should already exclude `.env`, `chroma_db/`, and uploaded documents.
2. Go to [share.streamlit.io](https://share.streamlit.io) and sign in with GitHub.
3. Click **New app** → select your repo → set main file to `app.py`.
4. Under **Advanced settings → Secrets**, paste:
   ```toml
   GEMINI_API_KEY = "your_real_key"
   GITHUB_TOKEN = "your_real_token"
   ```
5. Click **Deploy**. The first build takes a few minutes — it installs dependencies and downloads the embedding model.

> ⚠️ **Persistence caveat:** Streamlit Community Cloud's filesystem is ephemeral. `chroma_db/`, `users.json`, and uploaded documents reset on every restart/redeploy. This is fine for a demo, but not for a production deployment — see the troubleshooting table above for the fix.

---

### You're done ✅
Once the app is running, you can sign up, upload a PDF/TXT/MD/DOCX file, and start asking questions — every answer will come back with `[source: page]` citations you can verify against the original document.
